# Lab 8: Optical Character Recognition (OCR)

**Duration:** 2 hours 30 minutes. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
Labels, signs, parcels and forms carry **information in writing**: printed words and numbers. **OCR** (*optical character recognition*) is reading letters from a picture. In this lab the Jetson reads a printed label with a pretrained OCR engine (Tesseract), shows how sure it is about each word, and takes out a **date** and **numbers** from the text. You also learn how to prepare a picture so the reading gets better.

**By the end of this lab you can:**
1. Explain what OCR does and what its output is (words with a confidence).
2. Read text from a picture with a pretrained OCR engine.
3. Prepare a picture (grey, scale, threshold) and compare the OCR result before and after.
4. Pick useful data out of text with a pattern (regular expression) and save it in a log.
5. Test OCR on a real label and name the cases where it fails.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Privacy.** A label can hold a name or a number. Read only the labels you make or the labels your classmates agree to show. Do not read the documents of other people.

## 2. Background

**OCR.** Reading printed letters is harder than reading a QR code. The program must find the lines of text, cut out each letter and decide which character it is. This lab uses **Tesseract**, a free OCR engine that runs on the CPU. Its reading part is a small neural network that was trained on a lot of text, so you use a **pretrained** model. It gives you every word with a **confidence** from 0 to 100. A low confidence means the program is not sure.

| | QR or barcode (Lab 7) | OCR text (this lab) |
|---|---|---|
| Stored data | Text chosen by you, with error correction | Whatever is written |
| Method | Fixed pattern finder | Recognises letters |
| Speed | Fast, every picture | Slow (about half a second or more), so you press a button |
| Fails when | The code is blurry, small, shiny or cut | The letters are small, blurry, curly or on a busy background |

**Getting better OCR.** Tesseract works best with dark letters on a plain light background, about 30 pixels tall. You prepare the picture with the tools of Lab 6:

| Step | Why |
|---|---|
| Grey (`cv2.cvtColor`) | Colour does not help to read letters |
| Scale up x2 (`cv2.resize`) | Small letters become big enough |
| Threshold (`cv2.adaptiveThreshold`) | Makes letters black and the page white, also under uneven light |

**Patterns in text.** Often you do not need all the text, only a date or a number. A **regular expression** (regex) describes a pattern, for example `\d{2}/\d{2}/\d{4}` means "two digits, a slash, two digits, a slash, four digits". Python's `re.findall` finds every match.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(8)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (20 min)

### Step 1: Check the OCR Program
OCR needs the program **Tesseract**. Setup of a new Jetson installs it. This cell checks it.

In [ ]:
import glob
import os
import shutil
import subprocess
import sys

print("This notebook is saved in:", os.getcwd())
print("tesseract:", shutil.which("tesseract") or "MISSING")

If it says `MISSING`, install it now. The first cell asks for your password (it is not shown). The second cell installs the program (about 1 minute, it needs the internet).

In [ ]:
import labview
labview.sudo_login()

In [ ]:
!sudo apt install -y tesseract-ocr

### Step 2: Install the Python Part
`pytesseract` lets Python talk to Tesseract.

In [ ]:
for package, module in (("pytesseract", "pytesseract"),):
    try:
        __import__(module)
        print(package, "is installed.")
    except ImportError:
        print("Installing", package, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        print(package, "is installed.")

### Step 3: Load the Tools
Run the cell. It loads the tools for the whole lab and makes a place to keep the state of the live reader.

In [ ]:
import csv
import re
import time
from datetime import datetime

import cv2
import numpy as np
from IPython.display import Image, display

run = {"read": False, "words": [], "text": "", "note": ""}
print("Ready.")

### Step 4: Plan Your Reader
Think about where reading text helps.

| Idea | Reads | Data in the text |
|---|---|---|
| Medicine or food check | Printed label | Expiry date |
| Door sign | Printed sign | Room name |
| Parcel | Printed address label | Name and postcode |
| Machine | Printed serial plate | Serial number |
| Receipt | Printed receipt | Total amount |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Idea you chose:
- What text is read and where is the camera:
- What should the program do after it reads the text:

---
# Part B: Read Text with OCR (50 min)

### Step 1: OCR on a Picture
First read a picture you control. The cell makes a picture of a label with `cv2.putText`, so you know the right answer, and reads it with Tesseract.

In [ ]:
import pytesseract

label = np.full((200, 520, 3), 255, np.uint8)
cv2.putText(label, "MILK 1L", (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (0, 0, 0), 4)
cv2.putText(label, "EXP 14/03/2027", (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 0, 0), 3)
cv2.putText(label, "Lot 4821", (20, 185), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)
cv2.imwrite("label.png", label)
display(Image(filename="label.png"))

text = pytesseract.image_to_string(label)
print("Text read:")
print(text)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did OCR read all three lines correctly?:
- Which characters were wrong (if any)?:


### Step 2: Confidence of Each Word
`image_to_data` gives each word with a **box** and a **confidence**. The cell prints the words and keeps only the words with a confidence of at least `MIN_CONF`.

In [ ]:
MIN_CONF = 50            # words below this confidence (0 to 100) are ignored


def read_words(img):
    """Words found by OCR: a list of (word, confidence, x, y, w, h). Words with a low confidence are dropped."""
    data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
    words = []
    for i, word in enumerate(data["text"]):
        if word.strip() and float(data["conf"][i]) >= MIN_CONF:
            words.append((word.strip(), float(data["conf"][i]), data["left"][i], data["top"][i], data["width"][i], data["height"][i]))
    return words


for word, conf, x, y, w, h in read_words(label):
    print(f"{word:<12} confidence {conf:5.1f}   box ({x}, {y}, {w}, {h})")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which word has the lowest confidence?:
- What happens to the list when you set MIN_CONF = 90?:


### Step 3: Prepare the Picture
Real labels are not perfect. The function `prepare` turns a picture into black letters on white, scaled up. The cell makes a hard picture (grey letters, uneven light and noise), reads it **before** and **after** the preparation, and shows the difference.

In [ ]:
def prepare(img, scale=2):
    """Grey, bigger, and black-on-white with a threshold that adapts to uneven light."""
    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    grey = cv2.resize(grey, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    return cv2.adaptiveThreshold(grey, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)


rng = np.random.default_rng(1)
hard = (label * 0.45 + 90).astype(np.uint8)                      # grey letters on a grey page
gradient = np.tile(np.linspace(-50, 50, hard.shape[1]), (hard.shape[0], 1))[..., None]
hard = np.clip(hard + gradient + rng.normal(0, 12, hard.shape), 0, 255).astype(np.uint8)   # uneven light and noise
cv2.imwrite("label_hard.png", hard)
display(Image(filename="label_hard.png"))

print("Before preparing:", repr(pytesseract.image_to_string(hard).strip()))
print("After preparing: ", repr(pytesseract.image_to_string(prepare(hard)).strip()))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which text did OCR read before and after the preparation?:
- Which of the three steps (grey, scale, threshold) do you think helped the most? Why?:


### Step 4: Pick the Data out of the Text
The function `find_data` uses regular expressions to take out the **date** and the **numbers** and ignores the rest.

In [ ]:
def find_data(text):
    """The dates (dd/mm/yyyy) and the numbers of 3 or more digits found in a text."""
    dates = re.findall(r"\b\d{2}/\d{2}/\d{4}\b", text)
    numbers = [n for n in re.findall(r"\b\d{3,}\b", text) if n not in "".join(dates)]
    return dates, numbers


dates, numbers = find_data(pytesseract.image_to_string(prepare(label)))
print("Dates:  ", dates)
print("Numbers:", numbers)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Dates and numbers found:
- What does `\\d{2}` mean in the pattern?:
- Why is the pattern better than reading every line yourself?:


### Step 5: Read a Real Label Live
The program reads text only when you press **Read text**, because OCR is slow. Hold a printed label, a sheet of text or a box with writing in front of the camera, keep it still and flat, and press the button. The words are drawn in boxes (green: sure, orange: less sure) and the dates and numbers are saved in `ocr_log.csv`.

In [ ]:
def want_read(state):
    run["read"] = True                       # the button only sets a flag; process does the slow work


def log_ocr(dates, numbers, text):
    first = not os.path.exists("ocr_log.csv")
    with open("ocr_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "dates", "numbers", "text"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), " ".join(dates), " ".join(numbers), " ".join(text.split())[:200]])


def process_ocr(frame, state):
    shown = frame.copy()
    if run["read"]:
        run["read"] = False
        started = time.time()
        small = prepare(frame)
        words = read_words(small)
        run["words"] = [(w, c, x // 2, y // 2, ww // 2, hh // 2) for w, c, x, y, ww, hh in words]      # back to the size of the picture
        run["text"] = " ".join(w[0] for w in words)
        dates, numbers = find_data(run["text"])
        run["note"] = f"{len(words)} words in {time.time() - started:.1f} s | dates {dates} numbers {numbers}"
        log_ocr(dates, numbers, run["text"])
    for word, conf, x, y, w, h in run.get("words", []):
        colour = (0, 200, 0) if conf >= 75 else (0, 160, 255)
        cv2.rectangle(shown, (x, y), (x + w, y + h), colour, 2)
        cv2.putText(shown, word, (x, max(y - 4, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colour, 1)
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    if run["note"]:
        cv2.putText(shown, run["note"][:75], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return shown


print("The OCR program is ready.")

In [ ]:
run.update(read=False, words=[], text="", note="")
live = labview.live(process_ocr, actions={"Read text": want_read})

After you pressed Stop, see what was read:

In [ ]:
print("Last text read:")
print(run["text"])
print()
if os.path.exists("ocr_log.csv"):
    with open("ocr_log.csv") as f:
        print(f.read())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Time that one reading took (shown at the bottom of the picture):
- Which text was read well and which badly?:
- What changed when you moved the label closer, tilted it, or changed the light?:
- Which dates or numbers did the program find?:


---
# Part C: Make It Your Own (20 min)
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the live cell again.

- Read text in a second language by `pytesseract.image_to_string(img, lang="eng")` (other languages need their data installed).
- Make `find_data` find another pattern, for example a phone number or a price like `RM 12.50`.
- Show the **expiry check**: compare the date found with today's date and write `EXPIRED` or `OK` on the screen.
- Cut the picture to a **region of interest** (a box in the middle) so OCR reads only that part and is faster.
- Save the words with the lowest confidence in a second file, so you can see what the engine finds hard.

> **Tip.** Change one thing at a time and run it after each change.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change:
- Did it work the first time? What went wrong:

---
# Part D: Test and Present (25 min)

### Step 1: Test
Delete the old log, then test these situations and write what happened.

In [ ]:
if os.path.exists("ocr_log.csv"):
    os.remove("ocr_log.csv")
print("Clean. Files left:", sorted(glob.glob("ocr_*")))

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | A printed label, flat, good light | Words in green boxes, date found | |
| 2 | The same label, tilted | Lower confidence or wrong words | |
| 3 | The same label in dim light (cover the lamp) | Lower confidence or wrong words | |
| 4 | Small letters (move the camera back) | Fewer words or wrong words | |
| 5 | A label on a curved bottle | Read or not? | |
| 6 | Handwriting | Read or not? | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 3: what did you see:
- Test 4 to 6: what did you see:
- The best distance and light for your label:

### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration**: read a label with OCR, show the confidence of the words, show the date found and show the log.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Idea and what the program reads:
- The time of one OCR reading and the FPS:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:

> **Where next.** In Lab 9 you start the AI vision tasks of YOLO. They find *objects* in the picture, not letters. Later, in Lab 16, you can use OCR and the codes of Lab 7 together with the detectors.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | OCR reads the label picture and the confidence of each word is shown | |
| 2 | The prepared picture reads better than the original (Part B Step 3) | |
| 3 | A date and numbers are taken out of the text with a pattern | |
| 4 | The live OCR finds a date or a number from a real label | |
| 5 | At least one improvement works and you explain the code you changed | |
| 6 | The test table is complete (the cases where OCR fails are written) | |
| 7 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What is OCR? What is the output of Tesseract for one word?:
- 2. Why is OCR run only when you press a button?:
- 3. Name the three steps that prepare a picture for OCR, and say what each one does.:
- 4. What does the confidence of a word tell you? What is the cost of a high `MIN_CONF`?:
- 5. Write a regular expression for a time such as 09:45 (two digits, a colon, two digits).:
- 6. Why is OCR of a curved bottle harder than OCR of a flat label?:
- 7. Give one case where a QR code is a better choice than OCR, and one where OCR is the only choice.:
- 8. Which is a good use of OCR? (a) Finding a face (b) Reading the expiry date on a package (c) Counting cars (d) Measuring a temperature:
- 9. Tesseract was trained on a lot of text. Why does it still fail on handwriting?:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `tesseract: MISSING` or `TesseractNotFoundError` | Tesseract is not installed | Run the three install cells of Part A (password, then `apt install`) |
| `No module named 'pytesseract'` | Step 2 of Part A was not run | Run Step 2 of Part A |
| OCR gives nonsense | Small letters, busy background, bad light | Use a plain label, bigger letters, good light. Check `prepare` with different `scale` values. |
| `NameError: name 'np' is not defined` | Step 3 of Part A was not run | Run the cell that loads the tools (Step 3 of Part A) and then the cells below it |
| `Cannot open the camera` | Another program uses the camera, or the cable is loose | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| The Read text button does nothing | The live cell was not run, or the picture is not updating | Run the cell that starts the live view again, then press the button |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(8)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Tesseract OCR: https://github.com/tesseract-ocr/tesseract
- pytesseract: https://github.com/madmaze/pytesseract
- Python documentation, re (regular expressions): https://docs.python.org/3/library/re.html